## Librerias

In [32]:
import geopandas as gpd
import pandas as pd
import unicodedata

## Cargando ambos datos

In [33]:
df_organizacion_territorial = gpd.read_file(
    "ORGANIZACIÓN TERRITORIAL PARROQUIAL 03.02.2026/ORGANIZACION_TERRITORIAL_PARROQUIAL.shp"
)
df_organizacion_territorial.head(3)

,DPA_PARROQ,DPA_DESPAR,DPA_CANTON,DPA_DESCAN,DPA_PROVIN,DPA_DESPRO,DPA_ANIO,txt,geometry
0,010150,CUENCA,0101,CUENCA,01,AZUAY,2026,CABECERA CANTONAL,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,010151,BAÑOS,0101,CUENCA,01,AZUAY,2026,PARROQUIA RURAL,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,010152,CUMBE,0101,CUENCA,01,AZUAY,2026,PARROQUIA RURAL,"POLYGON ((719475.594 9663878.887, 719478.891 9..."


In [34]:
df_organizacion_territorial.shape

(1050, 9)

In [35]:
df_densidad_poblacional= pd.read_csv("densidad_POBLACIONAL_BASE.csv", usecols= ["Provincia", "Cantón", "Parroquia", "Densidad Poblacional"])
df_densidad_poblacional.head(3)

,Provincia,Cantón,Parroquia,Densidad Poblacional
0,AZUAY,CUENCA,CUENCA,5.044
1,AZUAY,CUENCA,BAÑOS,87.000
2,AZUAY,CUENCA,CUMBE,86.000


In [36]:
df_densidad_poblacional.shape

(1042, 4)

### Renombrando Columnas de densidad_poblacional

In [37]:
df_densidad_poblacional = df_densidad_poblacional.rename(columns={
    "Provincia": "DPA_DESPRO",
    "Parroquia": "DPA_DESPAR",
    "Cantón": "DPA_DESCAN"
})
df_densidad_poblacional.head(3)

,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,Densidad Poblacional
0,AZUAY,CUENCA,CUENCA,5.044
1,AZUAY,CUENCA,BAÑOS,87.000
2,AZUAY,CUENCA,CUMBE,86.000


## Normalizando columnas para unirlos mediante columnas

In [38]:
def normalizar(texto):

    if pd.isna(texto):
        return texto

    #Eliminar espacios al inicio y al final, y coloca en mayusculas
    texto = str(texto).strip().upper()

    # Eliminar tildes y diéresis
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Eliminar espacios dobles
    texto = ' '.join(texto.split())

    return texto

df_organizacion_territorial["DPA_DESPAR"] = df_organizacion_territorial["DPA_DESPAR"].apply(normalizar)
df_densidad_poblacional["DPA_DESPAR"] = df_densidad_poblacional["DPA_DESPAR"].apply(normalizar)

df_organizacion_territorial["DPA_DESPRO"] = df_organizacion_territorial["DPA_DESPRO"].apply(normalizar)
df_densidad_poblacional["DPA_DESPRO"] = df_densidad_poblacional["DPA_DESPRO"].apply(normalizar)

df_organizacion_territorial["DPA_DESCAN"] = df_organizacion_territorial["DPA_DESCAN"].apply(normalizar)
df_densidad_poblacional["DPA_DESCAN"] = df_densidad_poblacional["DPA_DESCAN"].apply(normalizar)

## Viendo filas faltantes

In [39]:
df_densidad_poblacional.columns

Index(['DPA_DESPRO', 'DPA_DESCAN', 'DPA_DESPAR', 'Densidad Poblacional'], dtype='object')

In [40]:
columnas = ['DPA_DESPRO', 'DPA_DESCAN', 'DPA_DESPAR']

df_faltantes = df_organizacion_territorial[
    ~df_organizacion_territorial.set_index(columnas).index.isin(df_densidad_poblacional.set_index(columnas).index)
]
print(len(df_faltantes))
df_faltantes


10


,DPA_PARROQ,DPA_DESPAR,DPA_CANTON,DPA_DESCAN,DPA_PROVIN,DPA_DESPRO,DPA_ANIO,txt,geometry
447,100153,LA CAROLINA,1001,IBARRA,10,IMBABURA,2026,PARROQUIA RURAL,"POLYGON ((798870.674 10089124.194, 798870.718 ..."
660,131251,SOSOTE,1312,ROCAFUERTE,13,MANABI,2026,PARROQUIA RURAL,"POLYGON ((556447.075 9904427.328, 556436.221 9..."
696,140190,"ZONA EN ESTUDIO ""SINAI-CUCHAENTZA""",1401,MORONA,14,MORONA SANTIAGO,2026,ZONA EN ESTUDIO,"POLYGON ((839410.478 9767712.318, 839415.887 9..."
746,141350,SEVILLA DON BOSCO,1413,SEVILLA DON BOSCO,14,MORONA SANTIAGO,2026,CABECERA CANTONAL,"POLYGON ((829941.106 9764498.83, 829969.204 97..."
786,160167,SHUAR PASTAZA,1601,PASTAZA,16,PASTAZA,2026,PARROQUIA RURAL,"POLYGON ((868732.145 9798802.56, 868733.81 979..."
835,170257,JUAN MONTALVO,1702,CAYAMBE,17,PICHINCHA,2026,PARROQUIA RURAL,"POLYGON ((835301.749 10002713.71, 835243.914 1..."
979,210456,LA MAGDALENA,2104,SHUSHUFINDI,21,SUCUMBIOS,2026,PARROQUIA RURAL,"POLYGON ((977506.338 9975323.769, 977514.637 9..."
980,210457,LA PRIMAVERA,2104,SHUSHUFINDI,21,SUCUMBIOS,2026,PARROQUIA RURAL,"POLYGON ((984425.686 9999306.515, 984486.011 9..."
1048,900651,ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO),9006,ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO),90,ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO),2026,ZONA EN ESTUDIO,"POLYGON ((765124.686 9745394.336, 765242.327 9..."
1049,ISLA,ISLA,ISLA,ISLA,ISLA,ISLA,2026,ISLA,"MULTIPOLYGON (((583954 9632994, 583959 9632989..."


Se ignora por que son poco datos y no hay densidad poblacional para aquellas parroquias

## Uniendo mediante columnas de "DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR" y agregando geometria a densidad poblacional

In [41]:
df_densidad_poblacional = df_densidad_poblacional.merge(
    df_organizacion_territorial[
        ["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR", "geometry"]
    ],
    on=["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR"],
    how="left"
)
df_densidad_poblacional.head(3)

,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,Densidad Poblacional,geometry
0,AZUAY,CUENCA,CUENCA,5.044,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,AZUAY,CUENCA,BANOS,87.000,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,AZUAY,CUENCA,CUMBE,86.000,"POLYGON ((719475.594 9663878.887, 719478.891 9..."


## Cargando nuevamente dataset de Organizacion territorial para convervar nombres de parroquias, cantones y provincias sin normalizar

In [42]:
df_organizacion_territorial = gpd.read_file(filename="ORGANIZACIÓN TERRITORIAL PARROQUIAL 03.02.2026/ORGANIZACION_TERRITORIAL_PARROQUIAL.shp")
df_organizacion_territorial.head(3)

,DPA_PARROQ,DPA_DESPAR,DPA_CANTON,DPA_DESCAN,DPA_PROVIN,DPA_DESPRO,DPA_ANIO,txt,geometry
0,010150,CUENCA,0101,CUENCA,01,AZUAY,2026,CABECERA CANTONAL,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,010151,BAÑOS,0101,CUENCA,01,AZUAY,2026,PARROQUIA RURAL,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,010152,CUMBE,0101,CUENCA,01,AZUAY,2026,PARROQUIA RURAL,"POLYGON ((719475.594 9663878.887, 719478.891 9..."


In [43]:
df_densidad_poblacional.head(3)

,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,Densidad Poblacional,geometry
0,AZUAY,CUENCA,CUENCA,5.044,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,AZUAY,CUENCA,BANOS,87.000,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,AZUAY,CUENCA,CUMBE,86.000,"POLYGON ((719475.594 9663878.887, 719478.891 9..."


## Eliminando "DPA_DESPRO", "DPA_DESCAN", "DPA_DESPAR" de densidad poblacional para colocar las de organizacion_territorial mediante geometria

In [44]:
df_densidad_poblacional = df_densidad_poblacional.drop(columns= ["DPA_DESPRO", "DPA_DESCAN", "DPA_DESPAR"])

## Tranformando densidad poblacional a geopandas

In [45]:
df_densidad_poblacional = gpd.GeoDataFrame(
    df_densidad_poblacional,
    geometry="geometry"
)
df_densidad_poblacional.head()

,Densidad Poblacional,geometry
0,5.044,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,87.000,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,86.000,"POLYGON ((719475.594 9663878.887, 719478.891 9..."
3,5.000,"POLYGON ((685768.602 9685990.681, 685768.699 9..."
4,29.000,"POLYGON ((716308.838 9708269.142, 718521.46 97..."


## Primer union, a el densidad poblacional (Que actualmente solo tiene geometria y densidad poblacional) se le agregan las columnas de parroquia, canton y provincia mediante su geometria

In [46]:
densidad_poblacional_final = df_densidad_poblacional.merge(
    df_organizacion_territorial[
        [
            "geometry",
            "DPA_DESPAR",
            "DPA_DESCAN",
            "DPA_DESPRO",
        ]
    ],
    on="geometry",
    how="left"
)
densidad_poblacional_final.head()

,Densidad Poblacional,geometry,DPA_DESPAR,DPA_DESCAN,DPA_DESPRO
0,5.044,"POLYGON ((724237.158 9687637.128, 724238.067 9...",CUENCA,CUENCA,AZUAY
1,87.000,"POLYGON ((714383.483 9679807.792, 714390.309 9...",BAÑOS,CUENCA,AZUAY
2,86.000,"POLYGON ((719475.594 9663878.887, 719478.891 9...",CUMBE,CUENCA,AZUAY
3,5.000,"POLYGON ((685768.602 9685990.681, 685768.699 9...",CHAUCHA,CUENCA,AZUAY
4,29.000,"POLYGON ((716308.838 9708269.142, 718521.46 97...",CHECA,CUENCA,AZUAY


In [47]:
df_organizacion_territorial.columns

Index(['DPA_PARROQ', 'DPA_DESPAR', 'DPA_CANTON', 'DPA_DESCAN', 'DPA_PROVIN',
       'DPA_DESPRO', 'DPA_ANIO', 'txt', 'geometry'],
      dtype='object')

## Se hace uso del densidad poblacional (que solo tiene densidad y geometria) y se une unicamente el codigo de parroquia mediante geometria de ambos

In [48]:
df_densidad_poblacional.head()

,Densidad Poblacional,geometry
0,5.044,"POLYGON ((724237.158 9687637.128, 724238.067 9..."
1,87.000,"POLYGON ((714383.483 9679807.792, 714390.309 9..."
2,86.000,"POLYGON ((719475.594 9663878.887, 719478.891 9..."
3,5.000,"POLYGON ((685768.602 9685990.681, 685768.699 9..."
4,29.000,"POLYGON ((716308.838 9708269.142, 718521.46 97..."


In [49]:
densidad_poblacional_final_2 = df_densidad_poblacional.merge(
    df_organizacion_territorial[
        [
            "geometry",
            "DPA_PARROQ",
        ]
    ],
    on="geometry",
    how="left"
)

In [50]:
densidad_poblacional_final_2 = densidad_poblacional_final_2.drop("geometry", axis= 1) #Eliminamos deometria
densidad_poblacional_final_2.head(2)

,Densidad Poblacional,DPA_PARROQ
0,5.044,010150
1,87.000,010151


In [51]:
densidad_poblacional_final_2 = densidad_poblacional_final_2.set_index('DPA_PARROQ') #Colocamos de indice el codigo de parroquia
densidad_poblacional_final_2.head(3)

,Densidad Poblacional
DPA_PARROQ,
010150,5.044
010151,87.000
010152,86.000


In [52]:
densidad_poblacional_final_2.to_csv("densidad_poblacional.csv")#Transformamos a csv